# Exploración de modelos predictivos

Antes de llegar al modelo final (Random Forest, ver `03_modelo_final.ipynb`), se exploraron
varios enfoques alternativos. Este notebook documenta ese proceso de iteración: qué se probó,
por qué no fue suficiente, y cómo eso llevó a la decisión final de enfoque.

**Resumen del proceso:**
1. Construcción de un índice de peligrosidad propio (crímenes por 100.000 habitantes)
2. Reducción de dimensionalidad (PCA) sobre variables socioeconómicas
3. Regresión lineal sobre el índice de peligrosidad → resultado insuficiente (R² = 0.30)
4. Intento de modelo por Community Area individual con muy pocas observaciones (5 años) → no robusto
5. Ampliación del histórico con datos de 2015-2019 vía API del Chicago Data Portal
6. **Conclusión**: se necesitan variables socioeconómicas con granularidad anual real
   (no solo un valor agregado por zona) → este hallazgo definió el enfoque del modelo final,
   desarrollado conjuntamente con Random Forest sobre variables anualizadas.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm

pd.set_option("display.max_columns", None)

## 1. Carga de datos base (partiendo del EDA)

In [3]:
df_analisis = pd.read_csv("../data/Proyecto Bootcamp.csv")

df_analisis["Date"] = pd.to_datetime(df_analisis["Date"], format="%m/%d/%Y %I:%M:%S %p")
df_analisis["Year"] = df_analisis["Date"].dt.year
df_analisis["Month"] = df_analisis["Date"].dt.month
df_analisis["Hour"] = df_analisis["Date"].dt.hour

df_analisis = df_analisis.dropna()
df_analisis = df_analisis[df_analisis["Community Area"] != 0.0]
df_analisis = df_analisis[df_analisis["Year"] != 2025]

df_analisis["Arrest_dummy"] = df_analisis["Arrest"].astype(int)
df_analisis["Domestic_dummy"] = df_analisis["Domestic"].astype(int)

df_analisis.shape

(1418334, 27)

## 2. Incorporación de variables socioeconómicas por Community Area

Fuente: [Chicago Health Atlas — Community Areas](https://igchicago.org/information-portal/data-dashboards/)

Variables incorporadas: renta per cápita (PCI), nivel educativo (HCSBDP), % población de color (POC) y población total (POP).

In [6]:
df_salud = pd.read_csv("../data/Chicago Health Atlas Data Download - Community areas.csv", header=0)
df_salud = df_salud.iloc[4:].reset_index(drop=True)

df_salud = df_salud[["GEOID", "PCI_2020-2024", "HCSBDP_2023-2024", "POC_2020-2024", "POP_2020-2024"]]
df_salud = df_salud.rename(columns={"GEOID": "Community Area"})
df_salud["Community Area"] = df_salud["Community Area"].astype(float)

df_analisis = df_analisis.merge(df_salud, on="Community Area", how="left")

for col in ["PCI_2020-2024", "HCSBDP_2023-2024", "POC_2020-2024", "POP_2020-2024"]:
    df_analisis[col] = pd.to_numeric(df_analisis[col], errors="coerce")

df_analisis.shape

(1418334, 31)

## 3. Construcción de un índice de peligrosidad propio

Idea inicial: normalizar el número de crímenes por población para obtener una métrica comparable entre Community Areas de tamaños muy distintos (crímenes por 100.000 habitantes).

In [7]:
indice = df_analisis.groupby("Community Area").size().reset_index(name="total_crimenes")
indice = indice.merge(
    df_analisis[["Community Area", "POP_2020-2024"]].drop_duplicates(),
    on="Community Area"
)
indice["indice_peligrosidad"] = indice["total_crimenes"] / indice["POP_2020-2024"] * 100000

df_analisis = df_analisis.merge(
    indice[["Community Area", "indice_peligrosidad"]], on="Community Area", how="left"
)
indice.head()

,Community Area,total_crimenes,POP_2020-2024,indice_peligrosidad
0,1.0,19932,54023.514047,36895.045337
1,2.0,17212,78390.308219,21956.795924
2,3.0,19798,54490.992635,36332.610295
3,4.0,9984,41565.112883,24020.144076
4,5.0,7761,36047.308966,21530.039891


## 4. Primer intento: PCA + Regresión Lineal sobre el índice de peligrosidad

In [8]:
# Reducción de dimensionalidad sobre variables socioeconómicas y dummies
X = df_analisis[[
    "PCI_2020-2024", "HCSBDP_2023-2024", "POC_2020-2024", "POP_2020-2024",
    "Arrest_dummy", "Domestic_dummy", "Hour", "Month", "Year"
]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
pca.fit(X_scaled)

varianza_acumulada = pca.explained_variance_ratio_.cumsum()
for i, va in enumerate(varianza_acumulada):
    print(f"Componente {i+1}: {va*100:.2f}% acumulado")

Componente 1: 32.25% acumulado
Componente 2: 44.99% acumulado
Componente 3: 56.31% acumulado
Componente 4: 67.28% acumulado
Componente 5: 77.67% acumulado
Componente 6: 87.55% acumulado
Componente 7: 94.79% acumulado
Componente 8: 98.61% acumulado
Componente 9: 100.00% acumulado


In [9]:
pca = PCA(n_components=6)
X_pca = pca.fit_transform(X_scaled)

X_train, X_test, y_train, y_test = train_test_split(
    X_pca, df_analisis["indice_peligrosidad"], test_size=0.2, random_state=42
)

modelo = LinearRegression()
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

print("R²:", round(r2_score(y_test, y_pred), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred)), 2))

R²: 0.3051
RMSE: 35518.89


**Resultado: R² ≈ 0.30.** El modelo explica muy poco de la variabilidad del índice de
peligrosidad. Esto se debe en parte a que se está prediciendo a nivel de crimen individual
(1.4M de filas) con variables que solo varían por Community Area, no por registro —
introduciendo mucho ruido.

## 5. Segundo intento: agregación por Community Area (una fila por zona)

In [10]:
df_modelo = df_analisis.groupby("Community Area").agg(
    PCI=("PCI_2020-2024", "mean"),
    HCSBDP=("HCSBDP_2023-2024", "mean"),
    POC=("POC_2020-2024", "mean"),
    POP=("POP_2020-2024", "mean"),
    Arrest=("Arrest_dummy", "mean"),
    Domestic=("Domestic_dummy", "mean"),
    indice_peligrosidad=("indice_peligrosidad", "first")
).reset_index()

df_modelo = df_modelo[df_modelo["Community Area"] != 0]
df_modelo.head()

,Community Area,PCI,HCSBDP,POC,POP,Arrest,Domestic,indice_peligrosidad
0,1.0,40324.957268,42.497201,55.097432,54023.514047,0.265703,0.145695,36895.045337
1,2.0,34945.225124,20.384129,60.123717,78390.308219,0.182082,0.135952,21956.795924
2,3.0,55561.676482,36.957608,48.406107,54490.992635,0.295535,0.090009,36332.610295
3,4.0,63201.960880,41.886330,37.921349,41565.112883,0.181090,0.093249,24020.144076
4,5.0,89577.649737,50.337589,26.431855,36047.308966,0.141992,0.061848,21530.039891


In [11]:
X = df_modelo[["PCI", "HCSBDP", "POC", "POP", "Arrest", "Domestic"]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
pca.fit(X_scaled)

varianza_acumulada = pca.explained_variance_ratio_.cumsum()
for i, va in enumerate(varianza_acumulada):
    print(f"Componente {i+1}: {va*100:.2f}% acumulado")

Componente 1: 59.02% acumulado
Componente 2: 76.51% acumulado
Componente 3: 85.70% acumulado
Componente 4: 93.44% acumulado
Componente 5: 96.85% acumulado
Componente 6: 100.00% acumulado


In [12]:
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

X_train, X_test, y_train, y_test = train_test_split(
    X_pca, df_modelo["indice_peligrosidad"], test_size=0.2, random_state=42
)

modelo = LinearRegression()
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

print("R²:", round(r2_score(y_test, y_pred), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred)), 2))

R²: 0.512
RMSE: 27149.08


**Mejora a R² ≈ 0.51** al agregar por zona, pero con solo 77 observaciones (una por Community Area) el modelo sigue siendo poco robusto.

In [13]:
df_modelo.corr()["indice_peligrosidad"].sort_values(ascending=False)

indice_peligrosidad    1.000000
Arrest                 0.766410
POC                    0.609262
Domestic               0.445645
Community Area         0.219279
POP                   -0.237282
HCSBDP                -0.246261
PCI                   -0.425336
Name: indice_peligrosidad, dtype: float64

## 6. Cambio de enfoque

Tras estos dos intentos, la conclusión fue la siguiente (razonamiento original del proceso):

> *"Pensándolo bien, no le veo sentido a hacerlo de esta forma, ya que al no tener variables
> de forma anual no puedo calcular nada con exactitud para 2025 y 2026, que son los objetivos
> del proyecto. Voy a realizar otro modelo predictivo simple solo con lo que tenemos (nº de
> crímenes y años), y después probaré añadiendo variables anuales para enriquecer el modelo."*

El problema de fondo: las variables socioeconómicas disponibles eran un **valor único agregado
por zona** (2020-2024), no una serie anual. Para predecir 2025/2026 con algo de rigor, se
necesitaban variables que evolucionaran año a año, no un promedio estático.


## 7. Modelo simple: tendencia lineal de crímenes por zona y año

In [14]:
crimenes_año = df_analisis.groupby(["Community Area", "Year"]).size().reset_index(name="crimenes")

predicciones = []
for area in crimenes_año["Community Area"].unique():
    datos = crimenes_año[crimenes_año["Community Area"] == area]
    X = datos[["Year"]]
    y = datos["crimenes"]
    modelo = LinearRegression().fit(X, y)
    for año in [2025, 2026]:
        pred = modelo.predict([[año]])[0]
        predicciones.append({"Community Area": area, "Year": año, "crimenes_pred": pred})

df_predicciones = pd.DataFrame(predicciones)
df_predicciones.head(10)

c:\Users\alejv\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\alejv\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\alejv\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\alejv\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\alejv\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py

,Community Area,Year,crimenes_pred
0,1.0,2025,2828.7
1,1.0,2026,2442.8
2,2.0,2025,2647.7
3,2.0,2026,2382.8
4,3.0,2025,2777.6
5,3.0,2026,2383.6
6,4.0,2025,1576.8
7,4.0,2026,1436.8
8,5.0,2025,1069.2
9,5.0,2026,908.2


In [15]:
# Validación con OLS para una zona concreta (ejemplo: Community Area 1)
area = 1.0
datos = crimenes_año[crimenes_año["Community Area"] == area]

X = sm.add_constant(datos["Year"])
y = datos["crimenes"]

modelo_ols = sm.OLS(y, X).fit()
print(modelo_ols.summary())

                            OLS Regression Results                            
Dep. Variable:               crimenes   R-squared:                       0.906
Model:                            OLS   Adj. R-squared:                  0.874
Method:                 Least Squares   F-statistic:                     28.82
Date:                Wed, 24 Jun 2026   Prob (F-statistic):             0.0127
Time:                        17:36:14   Log-Likelihood:                -32.949
No. Observations:                   5   AIC:                             69.90
Df Residuals:                       3   BIC:                             69.12
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       7.843e+05   1.45e+05      5.396      0.0

c:\Users\alejv\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\stats\stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 5 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


Con solo 5 años de histórico (2020-2024) por zona, los modelos individuales por Community Area tienen muy pocas observaciones para ser estadísticamente robustos.

## 8. Intento de enriquecer el histórico: datos de 2015-2019 vía API

Para tener más años de histórico por zona, se exploró ampliar el dataset descargando directamente los años 2015-2019 desde la API pública del Chicago Data Portal.

In [17]:
url = (
    "https://data.cityofchicago.org/resource/ijzp-q8t2.csv"
    "?$where=year%20between%202015%20and%202019&$limit=2000000"
)
df_historico = pd.read_csv(url)
print(df_historico.shape)
df_historico = pd.read_csv(url)
print(df_historico.shape)

(1335094, 22)
(1335094, 22)


In [18]:
df_historico = df_historico.rename(columns={
    "year": "Year",
    "community_area": "Community Area",
    "primary_type": "Primary Type",
    "location_description": "Location Description",
    "arrest": "Arrest",
    "domestic": "Domestic",
    "district": "District",
})

df_todosaños = pd.concat([df_historico, df_analisis], ignore_index=True)
df_todosaños = df_todosaños.dropna(subset=["Community Area"])
df_todosaños["Year"] = df_todosaños["Year"].astype(int)
df_todosaños = df_todosaños[df_todosaños["Year"] != 2025]

df_todosaños["Year"].value_counts().sort_index()

Year
2015    264887
2016    269970
2017    269300
2018    269148
2019    261705
2020    334396
2021    304267
2022    269329
2023    259610
2024    250732
Name: count, dtype: int64

In [19]:
crimenes_año_completo = df_todosaños.groupby(["Community Area", "Year"]).size().reset_index(name="crimenes")

crimenes_total_año = crimenes_año_completo.groupby("Year")["crimenes"].sum().reset_index()

X = sm.add_constant(crimenes_total_año["Year"])
y = crimenes_total_año["crimenes"]

modelo_ols_total = sm.OLS(y, X).fit()
print(modelo_ols_total.summary())

                            OLS Regression Results                            
Dep. Variable:               crimenes   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.125
Method:                 Least Squares   F-statistic:                  0.002039
Date:                Wed, 24 Jun 2026   Prob (F-statistic):              0.965
Time:                        18:12:31   Log-Likelihood:                -114.92
No. Observations:                  10   AIC:                             233.8
Df Residuals:                       8   BIC:                             234.4
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       5.412e+05   5.89e+06      0.092      0.9

c:\Users\alejv\AppData\Local\Programs\Python\Python310\lib\site-packages\scipy\stats\_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=10 observations were given.
  return hypotest_fun_in(*args, **kwds)


**Resultado:** incluso con 10 años de histórico total, la tendencia agregada por año
no resultó significativa (R² ≈ 0). Esto confirmó que el camino correcto no era ampliar el
histórico temporal sin más, sino **incorporar variables socioeconómicas con resolución
anual real** -- que es exactamente el enfoque que se desarrolló después, en colaboración,
para el modelo final con Random Forest (ver `03_modelo_final.ipynb`).


## Conclusiones de este proceso de exploración

| Enfoque probado | Resultado | Limitación principal |
|---|---|---|
| PCA + Regresión Lineal (nivel registro) | R² ≈ 0.30 | Variables de zona aplicadas a 1.4M de registros individuales |
| PCA + Regresión Lineal (nivel zona agregada) | R² ≈ 0.51 | Solo 77 observaciones (una por zona) |
| Tendencia lineal por zona y año | No robusto | Solo 5 años de histórico por zona |
| Ampliación a 10 años (2015-2024) | R² ≈ 0 | Variables socioeconómicas seguían sin resolución anual |

**Aprendizaje clave:** el cuello de botella no era el algoritmo, sino la **granularidad de
las variables socioeconómicas**. Este diagnóstico llevó al equipo a buscar variables
socioeconómicas anualizadas por Community Area, que es la base del modelo final
(Random Forest, R² = 0.977).
